# Value Signal Lab
## Valor do cliente e evidência experimental

**Estudo independente de portfólio com dados inteiramente sintéticos.** Nenhum resultado mede desempenho comercial real.

Pergunta: clientes mais baratos de adquirir também oferecem o maior retorno em 90 dias?

In [1]:
import pandas as pd
from IPython.display import display

from value_signal_lab.data import (
    TARGET,
    channel_summary,
    mature_customers,
    simulate_customers,
    validate_customers,
)
from value_signal_lab.experiment import analyze_experiment, plan_sample_size
from value_signal_lab.model import fit_value_model, temporal_split

SEED, AS_OF = 42, "2026-11-01"
historical = validate_customers(simulate_customers(12000, SEED, AS_OF), AS_OF)
mature = mature_customers(historical, AS_OF)
display(
    pd.Series(
        {
            "total": len(historical),
            "maduros": len(mature),
            "imaturos excluídos": len(historical) - len(mature),
        }
    )
)

total                 12000
maduros               11338
imaturos excluídos      662
dtype: int64

### 1. Distribuição e aquisição
Receita acumulada não é lucro ou LTV vitalício. Custos e retorno por canal são descritivos; diferenças não identificam efeito causal do canal.

In [2]:
display(
    mature[[TARGET, "first_order_revenue", "acquisition_cost"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(2)
)
channels = channel_summary(mature)
display(channels.round(2))
top20 = mature.nlargest(int(len(mature) * 0.2), TARGET)[TARGET].sum() / mature[TARGET].sum()
print(f"Os 20% de maior receita concentram {top20:.1%} do valor observado.")

,revenue_90d,first_order_revenue,acquisition_cost
count,11338.00,11338.00,11338.00
mean,111.78,40.15,32.17
std,122.89,21.38,12.66
min,5.04,5.04,7.26
50%,75.85,35.54,30.03
90%,231.62,67.02,49.06
95%,319.77,80.53,55.80
99%,581.29,110.89,70.64
max,3581.26,410.07,132.45


,channel,customers,revenue_90d,spend,mean_value_90d,cac,roas_90d
0,Google,4504,543985.90,183950.48,120.78,40.84,2.96
1,Meta,4124,462288.83,120922.08,112.10,29.32,3.82
2,TikTok,2710,261122.01,59822.59,96.35,22.07,4.36


Os 20% de maior receita concentram 51.3% do valor observado.


### 2. Validação temporal e calibração
Treino em 2024, calibração em abril–junho/2025, teste em outubro/2025–julho/2026. Embargos permitem observar os 90 dias antes da próxima etapa. Apenas informações disponíveis na primeira compra entram como features.

O modelo é comparado com primeira compra × multiplicador estimado no treino. Ajuste multiplicativo global é aprendido na calibração, sem consultar o teste.

In [3]:
metrics, scored, deciles = fit_value_model(mature, SEED)
display(pd.DataFrame({k: metrics[k] for k in ["baseline", "model"]}).round(3))
display(pd.DataFrame({"clientes": metrics["split_sizes"], "datas": metrics["split_dates"]}))
display(deciles.round(2))
print(
    f"Lift do decil superior: {metrics['top_decile_lift']:.2f}x. Métrica de ordenação, não efeito incremental."
)

,baseline,model
mae,60.326,56.757
rmse,98.753,91.192
r2,0.231,0.344
aggregate_bias_pct,2.897,7.652


,clientes,datas
train,4421,"[2024-01-01, 2024-12-31]"
calibration,1120,"[2025-04-01, 2025-06-30]"
test,3599,"[2025-10-01, 2026-07-31]"


,decile,predicted,observed,customers
0,1,36.25,36.29,360
1,2,52.38,51.00,360
2,3,64.25,60.86,360
3,4,75.83,69.94,360
4,5,88.37,86.80,360
5,6,100.69,100.82,359
6,7,115.96,113.03,360
7,8,138.10,144.71,360
8,9,181.55,164.40,360
9,10,325.03,266.81,360


Lift do decil superior: 2.44x. Métrica de ordenação, não efeito incremental.


### 3. Experimento independente e incerteza
O cenário assume +12% de receita **posterior à primeira compra** e +2% de custo no tratamento. Não usa o modelo treinado para selecionar clientes. Portanto, não demonstra que o modelo causou melhoria.

Métrica primária: soma(receita) / soma(custo). Bootstrap reamostra clientes preservando pares de receita/custo. Decisão única pré-definida: limite inferior do IC 95% acima de +5%, após diagnóstico de alocação.

In [4]:
experiment = validate_customers(
    simulate_customers(6000, SEED + 1, AS_OF, experiment=True), AS_OF, experiment=True
)
ab, lifts = analyze_experiment(experiment, n_boot=2000, seed=SEED)
display(
    pd.Series(
        {
            k: ab[k]
            for k in [
                "control_roas",
                "treatment_roas",
                "relative_lift",
                "ci95",
                "srm_pvalue",
                "mature_counts",
            ]
        }
    )
)
print(ab["decision"])

control_roas                                       3.4402
treatment_roas                                   3.521759
relative_lift                                    0.023708
ci95                                [-0.032632, 0.081786]
srm_pvalue                                       0.423469
mature_counts     {'control': 3031, 'value_signal': 2969}
dtype: object

Inconclusivo para adoção: o intervalo não sustenta ganho acima do limiar prático.


### 4. Planejar antes de testar
A aproximação para tamanho amostral usa variância da métrica de razão no histórico de treino, com poder de 80%, α bilateral de 5% e MDE relativo de 5% contra efeito zero. Isso não garante poder de 80% para que o limite inferior supere o limiar de adoção de 5%. O resultado é por braço e pressupõe clientes independentes, distribuições semelhantes e leitura com janela madura.

In [5]:
plan = plan_sample_size(temporal_split(mature)["train"])
display(pd.Series(plan))

customers_per_arm              7965.000000
minimum_relative_lift             0.050000
power                             0.800000
alpha                             0.050000
relative_influence_variance       1.268397
dtype: float64

### 5. Interpretação e próximos estudos
- Boa previsão não prova que um sinal melhora leilões. Isso exige intervenção randomizada real.
- Comparações entre canais podem refletir seleção de clientes e diferenças de mix.
- Clientes de campanhas reais podem ser dependentes; inferência deve respeitar a unidade de randomização.
- Não convertidos e todo o gasto elegível devem entrar no estimando de um teste de aquisição real.
- Próximos estudos: estabilidade fora do período, margem, calibração por segmento e experimentos por cluster.

Gráficos interativos completos: abra `reports/demo/report.html` localmente. Veja também [metodologia](../docs/methodology.md) e [resultados](../reports/demo/README.md).